### Triggers for per user schema tracking..

In [1]:
"""
// ON CREATE TRIGGER
{

  CREATE TRIGGER schema_oncreate_trigger ON CREATE
  AFTER COMMIT EXECUTE
  WITH createdVertices, createdEdges

  // create nodes
  CALL {
    WITH createdVertices
    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH createdVertices, schema
    UNWIND createdVertices AS newNodes
    UNWIND labels(newNodes) AS addedLabel
    WITH addedLabel, count(*) AS num_occurances, schema
    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})
    SET schema_node.count = coalesce(schema_node.count, 0) + num_occurances
  }

  // create node properities
  CALL {
    WITH createdVertices
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND createdVertices AS addedNode
    With addedNode, labels(addedNode) AS nodeLabels, keys(addedNode) AS nodeKeys, schema
    // group keys and values to set for a label across all created nodes.
    UNWIND nodeLabels AS label
    UNWIND nodeKeys AS key
    WITH label, key, valuetype(addedNode[key]) AS propType, schema
    // for each label and key, get the first type for that property key because across all nodes they might have same label, same key, but different property type (very unlikely tho).
    WITH label, key, count(*) AS propCount, head(collect(propType)) AS propTypeHeads, schema
    // Now group by label, with their keys, and property type for those keys.
    WITH label, collect(key) AS propNames, collect(propTypeHeads) AS propTypes, collect(propCount) AS propCounts, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode)
    WHERE schema_node.label=label

    UNWIND range(0, size(propNames) - 1) AS i
    MERGE (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propNames[i]})
    // Only set property type if there is no type already (first time creating that property.)
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) + propCounts[i], schema_node_prop.type = coalesce(schema_node_prop.type, propTypes[i])
  }

  // create relationships
  CALL {
    WITH createdEdges
    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH createdEdges, schema
    UNWIND createdEdges AS newEdge
    WITH type(newEdge) AS edgeType, schema
    WITH edgeType, count(*) as num_occurances, schema
    MERGE (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship{type: edgeType})
    SET schema_rel.count = coalesce(schema_rel.count, 0) + num_occurances
  }

  // create relationship properities
  CALL {
    WITH createdEdges
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND createdEdges AS addedEdge
    WITH addedEdge, type(addedEdge) AS edgeType, keys(addedEdge) AS edgeKeys, schema
    // group keys and values to set for a type across all created nodes.
    UNWIND edgeKeys AS key
    WITH edgeType, key, valuetype(addedEdge[key]) AS propType, schema
    // for each type and key, get the first type for that property key because across all edges they might have same type, same key, but different property type (very unlikely tho).
    WITH edgeType, key, count(*) AS propCount, head(collect(propType)) AS propTypeHeads, schema
    // Now group by label, with their keys, and property type for those keys.
    WITH edgeType, collect(key) AS propNames, collect(propTypeHeads) AS propTypes, collect(propCount) AS propCounts, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship)
    WHERE schema_rel.type=edgeType

    UNWIND range(0, size(propNames) - 1) AS i
    MERGE (schema_rel)-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(schema_rel_prop :SchemaRelationshipProperty{property_name: propNames[i]})
    // Only set property type if there is no type already (first time creating that property.)
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) + propCounts[i], schema_rel_prop.type = coalesce(schema_rel_prop.type, propTypes[i])
  }

  // create node out_relationships
  CALL {
    WITH createdEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH createdEdges, schema
    UNWIND createdEdges AS newEdge
    WITH startNode(newEdge) AS startNode, type(newEdge) AS edgeType, endNode(newEdge) AS endNode, schema
    WITH startNode, edgeType, count(*) AS num_out_rel, collect(endNode) AS endNodes, schema
    UNWIND endNodes AS endNode
    UNWIND labels(endNode) AS endNodeLabel
    WITH startNode, edgeType, num_out_rel, collect(endNodeLabel) AS endNodeLabels, schema

    UNWIND labels(startNode) AS label
    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode)
    WHERE schema_node.label=label

    MERGE (schema_node)-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel :SchemaNodeOutRelationship{rel_type: edgeType})
    SET schema_node_out_rel.count = coalesce(schema_node_out_rel.count, 0) + num_out_rel, schema_node_out_rel.to_labels = collections.union(coalesce(schema_node_out_rel.to_labels, []), endNodeLabels)
  }

}


// ON UPDATE TRIGGER
{

  CREATE TRIGGER schema_onupdate_trigger ON UPDATE
  AFTER COMMIT EXECUTE
  WITH removedVertexProperties, removedEdgeProperties, setVertexLabels, removedVertexLabels

  // Reduce count of schema node properties and delete if reduced to 0
  CALL {
    WITH removedVertexProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedVertexProperties, schema
    UNWIND removedVertexProperties AS removedVertexProperty
    WITH removedVertexProperty.vertex AS node, removedVertexProperty.key AS removedPropertyKey, schema
    UNWIND labels(node) AS nodeLabel
    WITH nodeLabel, collect(removedPropertyKey) as removedPropertyKeys, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: nodeLabel})
    UNWIND removedPropertyKeys AS removedPropertyKey
    
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: removedPropertyKey})
    // reduce the count of that schema_node_prop
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - 1
  }

  // Reduce count of schema edge properties and delete if reduced to 0
  CALL {
    WITH removedEdgeProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedEdgeProperties, schema
    UNWIND removedEdgeProperties AS removedEdgeProperty
    WITH removedEdgeProperty.edge AS edge, removedEdgeProperty.key AS removedPropertyKey, schema
    WITH type(edge) AS edgeType, collect(removedPropertyKey) AS removedPropertyKeys, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship{type: edgeType})
    UNWIND removedPropertyKeys AS removedPropertyKey
    
    MATCH (schema_rel)-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(schema_rel_prop :SchemaRelationshipProperty{property_name: removedPropertyKey})
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) - 1
  }

  // Delete node property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH removedVertexProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedVertexProperties, schema
    UNWIND removedVertexProperties AS removedVertexProperty
    WITH removedVertexProperty.vertex AS node, schema
    UNWIND labels(node) AS editedNodeLabel
    WITH DISTINCT editedNodeLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(:SchemaNode {label: editedNodeLabel})-[:IS_SCHEMA_NODE_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // Delete relationship property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH removedEdgeProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedEdgeProperties, schema
    UNWIND removedEdgeProperties AS removedEdgeProperty
    WITH type(removedEdgeProperty.edge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(:SchemaRelationship{type: edgeType})-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // when additional label was set to vertex
  CALL {
    WITH setVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH setVertexLabels, schema
    UNWIND setVertexLabels AS labelSetOnVertex
    WITH labelSetOnVertex.label AS addedLabel, labelSetOnVertex.vertices AS listOfVertices, schema
    UNWIND listOfVertices AS singleVertex

    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})
    SET schema_node.count = coalesce(schema_node.count, 0) + 1

    WITH addedLabel, singleVertex, schema_node
    UNWIND keys(singleVertex) as propertyKey

    WITH addedLabel, singleVertex, propertyKey as propertyName, valueType(singleVertex[propertyKey]) AS propertyType, schema_node
    MERGE (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propertyName})
    // Only set property type if there is no type already (first time creating that property.)
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) + 1, schema_node_prop.type = coalesce(schema_node_prop.type, propertyType)

  }

  // when a label was removed from vertex
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, removedVertexLabel.vertices AS listOfVertices, schema
    UNWIND listOfVertices AS singleVertex

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: removedLabel})
    SET schema_node.count = schema_node.count - 1

    WITH removedLabel, singleVertex, schema_node
    UNWIND keys(singleVertex) as propertyKey

    WITH removedLabel, singleVertex, propertyKey as propertyName, valueType(singleVertex[propertyKey]) AS propertyType, schema_node
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propertyName})
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - 1
  }

  // Delete label, properities & out_relationships if the label count is less than 1 (zero)
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: removedLabel})
    WHERE schema_node.count < 1
    OPTIONAL MATCH (schema_node)-->(prop_or_rel) 
    DETACH DELETE prop_or_rel, schema_node
  }    

  // Delete label property if removing the label from a node reduced the count below 1 (zero)
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})-[:IS_SCHEMA_NODE_PROPERTY]->(prop) 
    WHERE prop.count < 1
    DETACH DELETE prop
  }

}


// ON DELETE TRIGGER
{

  CREATE TRIGGER schema_ondelete_trigger ON DELETE
  AFTER COMMIT EXECUTE
  WITH deletedVertices, deletedEdges

  // reduce node properities count.
  CALL {
    WITH deletedVertices
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedVertices as removedNode
    With labels(removedNode) AS nodeLabels, keys(removedNode) AS nodeKeys, schema
    // group keys and values to set for a label across all created nodes.
    UNWIND nodeLabels AS label
    UNWIND nodeKeys AS key
    WITH label, key, count(*) as propCount, schema
    // Now group by label, with their keys, and counts.
    WITH label, collect(key) as propNames, collect(propCount) as propCounts, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode)
    WHERE schema_node.label=label

    UNWIND range(0, size(propNames) - 1) AS i
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop {property_name: propNames[i]})
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - propCounts[i]

  }

  // Delete node property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedVertices
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels AS label
    WITH DISTINCT label, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: label})-[:IS_SCHEMA_NODE_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // reduce node out relationships count.
  CALL {
    WITH deletedVertices
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels as nodeLabel
    WITH nodeLabel, count(*) as num_label, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: nodeLabel})-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel)
    SET schema_node_out_rel.count = coalesce(schema_node_out_rel.count, 0) - num_label
  }

  // Delete node out relationship if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedVertices
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels as nodeLabel
    WITH DISTINCT nodeLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: nodeLabel})-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel)
    WHERE schema_node_out_rel.count < 1
    DETACH DELETE schema_node_out_rel
  }

  // reduce label count from deleted node.
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH deletedVertices, schema

    UNWIND deletedVertices AS removedNodes
    UNWIND labels(removedNodes) AS removedLabel
    WITH removedLabel, count(*) as num_occurances, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})
    SET schema_node.count = schema_node.count - num_occurances
  }

  // delete nodes is there count is less than 1 from reduction above
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH deletedVertices, schema
    UNWIND deletedVertices AS removedNodes
    UNWIND labels(removedNodes) AS removedLabel

    WITH DISTINCT removedLabel, schema
    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})
    WHERE schema_node.count < 1
    OPTIONAL MATCH (schema_node)-->(prop_or_rel) 
    DETACH DELETE prop_or_rel, schema_node
  }

  // reduce relationship properities count
  CALL {
    WITH deletedEdges
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedEdges as removedEdge
    WITH type(removedEdge) AS edgeType, keys(removedEdge) AS edgeKeys, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel)
    WHERE schema_rel.type=edgeType

    UNWIND edgeKeys AS key
    WITH edgeType, key, count(*) as propCount, schema_rel

    MATCH (schema_rel)-->(schema_rel_prop {property_name: key})
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) - propCount
  }


  // Delete relationship property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedEdges
    // get schema
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedEdges as removedEdge
    WITH type(removedEdge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->({type: edgeType})-->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }


  // reduce relationship count.
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH deletedEdges, schema
    UNWIND deletedEdges AS deletedEdge

    WITH type(deletedEdge) AS edgeType, schema
    WITH edgeType, count(*) as num_occurances, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel {type: edgeType})
    SET schema_rel.count = coalesce(schema_rel.count, 0) - num_occurances
  }

  // delete relationships if reducutions cause it to be less than 1.
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH deletedEdges, schema
    UNWIND deletedEdges AS deletedEdge

    WITH type(deletedEdge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel {type: edgeType})
    WHERE schema_rel.count < 1
    OPTIONAL MATCH (schema_rel)-->(rel_prop) 
    DETACH DELETE rel_prop, schema_rel
  }

}
"""

"\n// ON CREATE TRIGGER\n{\n\n  CREATE TRIGGER schema_oncreate_trigger ON CREATE\n  AFTER COMMIT EXECUTE\n  WITH createdVertices, createdEdges\n\n  // create nodes\n  CALL {\n    WITH createdVertices\n    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n    WITH createdVertices, schema\n    UNWIND createdVertices AS newNodes\n    UNWIND labels(newNodes) AS addedLabel\n    WITH addedLabel, count(*) AS num_occurances, schema\n    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})\n    SET schema_node.count = coalesce(schema_node.count, 0) + num_occurances\n  }\n\n  // create node properities\n  CALL {\n    WITH createdVertices\n    // get schema\n    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n    UNWIND createdVertices AS addedNode\n    With addedNode, labels(addedNode) AS nodeLabels, keys(addedNode) AS nodeKeys, schema\n    // group keys and values to set for a label across all created nodes.\n    UNWIND nodeLabels AS label\n    UNWI